# RGB resolution × stroke-budget ablation — validation only

The repository is public. This notebook clones it directly with no token and no bundle, checks out exact implementation commit `da1f10c`, verifies critical Git blobs, reruns all 184 tests, validates the five source hashes and all 94 baseline artifacts, and creates no experiment output. Conditions B/C/D are not executed here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('CELL 1 COMPLETE — DRIVE MOUNTED')

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'rgb-resolution-budget-ablation'
EXPECTED_COMMIT = 'da1f10ceb5b0501479f0037da40b00ccb8ad122b'
EXPECTED_BLOBS = {
    'src/latent_stroke_dynamics/rgb_coarse_to_fine.py': 'f74c488714a5f76784d8238e2faf6f0378bff62c',
    'src/latent_stroke_dynamics/rgb_resolution_budget_ablation.py': '5abc1c2886aa9991a225de5e69c4b0353de46c3e',
    'tests/test_rgb_resolution_budget_ablation.py': '7d73363f2a52569475a3b9f660ec24814ab9a877',
}

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
completed = subprocess.run(
    [
        'git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR),
    ],
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError('Public repository clone failed.')
subprocess.run(
    ['git', 'checkout', '--quiet', '--detach', EXPECTED_COMMIT],
    cwd=REPO_DIR,
    check=True,
)
observed_commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
assert observed_commit == EXPECTED_COMMIT, observed_commit
for path, expected_blob in EXPECTED_BLOBS.items():
    observed_blob = subprocess.check_output(
        ['git', 'rev-parse', f'HEAD:{path}'], cwd=REPO_DIR, text=True
    ).strip()
    assert observed_blob == expected_blob, (path, observed_blob)
assert subprocess.check_output(
    ['git', 'status', '--short'], cwd=REPO_DIR, text=True
).strip() == ''
print('CELL 2 COMPLETE — PUBLIC REPOSITORY CLONED; EXACT SOURCE PINNED', observed_commit)

In [ ]:
import subprocess

subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', '.'],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    check=True,
)
print('CELL 3 COMPLETE — 184 TESTS PASSED IN COLAB')

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import os
import shutil

TARGET_DIR = Path('/content/rgb-fixed-targets')
if TARGET_DIR.exists():
    shutil.rmtree(TARGET_DIR)
TARGET_DIR.mkdir()
expected_hashes = {
    'mercedes-benz-logo-black.jpg': '94ed621d3f74ec56cddd1d59e6536dca0c61adb41c87e5e9d6510765656ad2a7',
    'slice-vb00CqFW3YA-unsplash.jpg': '159d5d7ed97542093a513f37097513f2316a146b00b7d1b289e1ce8de4b7fa1a',
    'painted-vase-ideas-1.jpg': '429104e33e08e7998b0dcfc656873be6f0356cc72627be93ceb94b07700b3254',
    'w1200_9d41_iPhone_14_Pro_Max_-_Telephoto_Lens_-_After_-1_copy__1_.jpg': 'da60fea3d30ee85605a93537e9d6cf8fe5d12af62bb1dd778956eef55dde6b06',
    'anor-londo-sq.jpg': 'e9dc2de55d1a5fad6a6018de7acf681df8a6a632bfe394845cc84ea88003db52',
}
os.chdir('/content')
for filename in expected_hashes:
    loose_path = Path('/content') / filename
    if loose_path.exists():
        loose_path.unlink()
uploaded_targets = files.upload()
assert set(uploaded_targets) == set(expected_hashes), sorted(uploaded_targets)
for filename, expected_hash in expected_hashes.items():
    payload = uploaded_targets[filename]
    observed_hash = hashlib.sha256(payload).hexdigest()
    assert observed_hash == expected_hash, (filename, observed_hash)
    (TARGET_DIR / filename).write_bytes(payload)
    loose_path = Path('/content') / filename
    if loose_path.exists():
        loose_path.unlink()
print('CELL 4 COMPLETE — FIVE EXACT TARGETS VERIFIED')

In [ ]:
from pathlib import Path
import json
import subprocess

BASELINE_DIR = Path('/content/drive/MyDrive/latent-stroke-dynamics-rgb/rgb-coarse-to-fine-fixed-seed73')
OUTPUT_DIR = Path('/content/drive/MyDrive/latent-stroke-dynamics-rgb/rgb-resolution-budget-ablation-seed73')
INCOMPLETE_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '.incomplete')
VALIDATION_JSON = Path('/content/rgb_ablation_validation.json')
assert BASELINE_DIR.is_dir(), BASELINE_DIR
assert not OUTPUT_DIR.exists(), OUTPUT_DIR
assert not INCOMPLETE_DIR.exists(), INCOMPLETE_DIR
completed = subprocess.run(
    [
        'python',
        str(REPO_DIR / 'run_rgb_resolution_budget_ablation.py'),
        '--validate-only',
        '--input-dir', str(TARGET_DIR),
        '--baseline-dir', str(BASELINE_DIR),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError('Validation-only command failed.')
VALIDATION_JSON.write_text(completed.stdout, encoding='utf-8')
validation = json.loads(completed.stdout)
print('CELL 5 COMPLETE — VALIDATION REPORT CREATED IN /content ONLY')

In [ ]:
import subprocess

assert validation['status'] == 'rgb_resolution_budget_ablation_valid_no_outputs'
assert validation['output_side_effects'] is False
assert validation['training_performed'] is False
assert validation['learned_model_used'] is False
assert validation['frozen_phase_b0_decision_changed'] is False
assert validation['target_validation']['target_set_sha256'] == '31e1fcc2bf344f8b72d3f04dfbc9109c61c39fc8cbc10668c1b78d575a673b42'
assert validation['baseline_verification']['aggregate_summary_sha256'] == '7f2c32cef077bef1737a3f00ee584cf1075b1feb5711995ab87dd9812f233c05'
assert validation['baseline_verification']['verified_artifact_count'] == 94
assert abs(validation['baseline_verification']['mean_512_mse'] - 0.012319037602067864) < 1e-15
conditions = validation['protocol']['conditions']
assert [item['condition_id'] for item in conditions] == ['A', 'B', 'C', 'D']
assert [item['planning_size'] for item in conditions] == [96, 96, 128, 128]
assert [item['total_strokes'] for item in conditions] == [210, 420, 210, 420]
assert [item['execute'] for item in conditions] == [False, True, True, True]
assert all(item['deterministic'] for item in validation['synthetic_smoke'])
assert all(item['monotonic'] for item in validation['synthetic_smoke'])
assert not OUTPUT_DIR.exists()
assert not INCOMPLETE_DIR.exists()
worktree_status = subprocess.check_output(
    ['git', 'status', '--short'], cwd=REPO_DIR, text=True
).strip()
assert worktree_status == '', worktree_status
print('CELL 6 COMPLETE — PUBLIC-CLONE VALIDATION-ONLY GATE PASSED')
print('status:', validation['status'])
print('baseline artifacts verified:', validation['baseline_verification']['verified_artifact_count'])
print('conditions:', [
    (item['condition_id'], item['planning_size'], item['total_strokes'], item['execute'])
    for item in conditions
])
print('output directory exists:', OUTPUT_DIR.exists())
print('incomplete directory exists:', INCOMPLETE_DIR.exists())
print('training performed:', validation['training_performed'])
print('learned model used:', validation['learned_model_used'])
print('Phase B0 changed:', validation['frozen_phase_b0_decision_changed'])
print('git status clean:', worktree_status == '')